# Football Managerial Decision Analytics — Chapter 04
## Starting XI and substitution decision support

> **Data.** StatsBomb Open Data (https://github.com/hudl/open-data), provided by StatsBomb / Hudl, used for educational, non-commercial research. Raw data are downloaded from the official repository at run time and are not redistributed here. Every figure carries its chart ID from `docs/VISUALIZATION_CATALOGUE.md`.

This notebook is one chapter of the research record. All logic lives in the shared `src/` package; the cells call it. The same charts are available interactively in the Streamlit dashboard (`streamlit run app.py`).

In [ ]:
# Setup — locate (or fetch) the project code, import the shared package, load the workspace
import os, subprocess, sys, time
from pathlib import Path

REPO_URL = "https://github.com/blazingarrows1525/football-managerial-decision-analytics"
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "workspace.py").exists()), None)
if ROOT is None:  # opened on its own (e.g. Google Colab): fetch the project code once, next to this notebook
    ROOT = Path.cwd() / "football-managerial-decision-analytics"
    if not (ROOT / "src" / "workspace.py").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("FMA_PROJECT_ROOT", str(ROOT))  # data cache and outputs live inside the project folder

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from src.reporting.notebook_tools import (catalogue, explorer, implication, interpret, limitation, show, start,
                                          timed, timings)

pd.set_option("display.max_columns", 40, "display.width", 200, "display.float_format", "{:.3f}".format)
RUN_ADVANCED_GALLERY = True   # heavier, optional sections; the core story does not depend on them
SEED = 42                             # every resampling and model step takes this seed explicitly

ws = start(seed=SEED)                 # first run downloads and parses; later runs read the processed cache
B, T, FOCUS = ws.bundle, ws.tables, ws.focus
tm = ws.team_matches()                # the focus team's matches in date order
print(f"Advanced sections: {'on' if RUN_ADVANCED_GALLERY else 'off'} · Python {sys.version.split()[0]} · pandas {pd.__version__}")

## F · Starting XI decision support

**Research question.** Which eleven does the evidence available *before* a match shortlist for a formation, why
is each player in it, and how sensitive is that to the weights?

**Data and method.** Candidates are players seen in recent match-day squads before the decision point. Each
candidate–slot score is

`(0.40·role fit + 0.20·recent form + 0.15·evidence volume + 0.15·minutes-based exposure + 0.10·continuity) × position compatibility`

with components on 0–100. Players are assigned to slots with the Hungarian algorithm under hard constraints
(eleven distinct players, one goalkeeper, eligibility, squad membership). **It is a transparent heuristic:
"best" means best under these weights and constraints.**

In [ ]:
from src.analytics import lineup_optimizer as xi
mid = ws.default_match()
p = ws.params()
ctx = ws.xi_context(FOCUS, mid)
res = ws.xi_result(FOCUS, mid, p.formation)
print(f"Decision point: before {ws.match_label(mid)} · formation {p.formation}")
print("Pool:", ctx.pool_description)
display(pd.DataFrame([B.config.xi_score_weights]))
show("G-149", ws, match_id=mid)
show("G-166", ws, match_id=mid)
display(res.lineup[["slot", "player_name", "primary_role", "compatibility", "role_fit", "recent_form", "sample_reliability",
                    "workload_suitability", "lineup_continuity", "score", "coverage_label"]].round(1))
print("Optimiser warnings:", res.warnings or "none")

In [ ]:
cmp_, share = xi.compare_with_actual(res, T["appearances"], mid, ws.team_id())
show("G-150", ws, match_id=mid)
show("G-169", ws, match_id=mid)
only_model = cmp_[cmp_.in_recommended & ~cmp_.in_actual].player_name.tolist()
only_actual = cmp_[~cmp_.in_recommended & cmp_.in_actual].player_name.tolist()
interpret(f"{share:.0%} of the players who actually started are in the shortlist. Shortlisted but not selected: "
          f"{', '.join(only_model) or 'none'}; selected but not shortlisted: {', '.join(only_actual) or 'none'}.")
limitation("Agreement measures similarity to the manager's choice, not whether either was better. The source has "
           "no injury, fitness or training information, so a player the manager could not pick may still be shortlisted.")

In [ ]:
# Sensitivity: does the shortlist change when the weights change?
import dataclasses
base_ids = set(res.lineup.player_id)
rows = []
for name, w in {"configured": B.config.xi_score_weights,
                "role fit only": {"role_fit": 1, "recent_form": 0, "sample_reliability": 0, "workload_suitability": 0, "lineup_continuity": 0},
                "form-led": {"role_fit": .2, "recent_form": .5, "sample_reliability": .1, "workload_suitability": .1, "lineup_continuity": .1},
                "continuity-led": {"role_fit": .2, "recent_form": .1, "sample_reliability": .1, "workload_suitability": .1, "lineup_continuity": .5},
                "rotation-led": {"role_fit": .3, "recent_form": .1, "sample_reliability": .1, "workload_suitability": .5, "lineup_continuity": 0}}.items():
    r2 = xi.optimize_xi(ctx, p.formation, dataclasses.replace(B.config, xi_score_weights=w))
    ids = set(r2.lineup.player_id)
    rows.append({"weighting": name, "players kept from configured XI": len(ids & base_ids), "total score": round(r2.total_score, 1),
                 "new players": ", ".join(r2.lineup[~r2.lineup.player_id.isin(base_ids)].player_name) or "—"})
sens = pd.DataFrame(rows)
display(sens)
show("G-167", ws, match_id=mid)
interpret(f"Across five weightings between {sens['players kept from configured XI'].min()} and 11 players stay the same. "
          f"The slots that change are the ones where the evidence does not separate the candidates.")
implication("The tool is most useful as a way to see which slots are genuinely open and what each weighting "
            "assumes — not as a single answer.")

In [ ]:
# Optional, heavier section — set RUN_ADVANCED_GALLERY = True in the setup cell to run it
if RUN_ADVANCED_GALLERY:
    show("G-151", ws, match_id=mid)
    show("G-168", ws, match_id=mid)
    show("D-103", ws)
    show("G-165", ws)

In [ ]:
# Backtest: rebuild the shortlist before every match from earlier matches only, compare with the actual XI
with timed("XI backtest"):
    ag = ws.xi_agreement()
show("L-267", ws)
interpret(f"Over {len(ag)} matches the shortlist shares {ag.agreement_pct.mean():.1f}% of the actual XI on average "
          f"(minimum {ag.agreement_pct.min():.0f}%, maximum {ag.agreement_pct.max():.0f}%).")
limitation("High agreement reflects a stable core that any sensible rule would pick; it is not evidence of "
           "decision quality.")

## Substitution intelligence

**Research question.** When and in which situations are changes made, who replaces whom, and which bench
options does the evidence shortlist at a given moment?

**Method.** Each substitution is enriched with the score state from the reconstructed goal timeline, the
manpower state from sending-offs, a timing band, the roles of both players and the substitution window under a
configured five-substitutes / three-windows rule. Historical patterns are **descriptive**; scenario shortlists
are **counterfactual simulations**.

In [ ]:
S = B.subs; mine = S[S.team_name == FOCUS]; others = S[S.team_name != FOCUS]
show("H-171", ws)
show("H-174", ws)
show("H-173", ws)
interpret(f"{FOCUS} made {len(mine)} substitutions ({len(mine) / len(tm):.2f} per match) at a median minute of {mine.minute.median():.0f}; "
          f"{(mine.game_state == 'leading').mean():.0%} came while leading and {mine.like_for_like.astype(bool).mean():.0%} were "
          f"like-for-like by role. Opponents: median minute {others.minute.median():.0f}, {(others.game_state == 'trailing').mean():.0%} while trailing.")
limitation("Score state is driven by team strength: a team that usually leads makes most changes while leading "
           "whatever its intent. A substitution alone says nothing about injury or fatigue.")

In [ ]:
from src.analytics import substitution_analysis as sa
minute = 65.0
rec = ws.sub_scenario(FOCUS, mid, minute)
cx = rec["context"]
print(f"{ws.match_label(mid)} at {minute:.0f}': {cx['team_score']}-{cx['opponent_score']} ({cx['game_state']}), "
      f"{cx['subs_used']} substitutions used, {cx['subs_remaining']} remaining · intent '{rec['intent']}' ({rec.get('intent_reason', '')})")
print("Rule source:", cx["rule_source"])
show("H-180", ws, match_id=mid, minute=minute)
show("H-181", ws, match_id=mid, minute=minute)
show("H-182", ws, match_id=mid, minute=minute)
if rec["status"] == "ok":
    on = rec["recommended_on"]
    interpret(f"The heuristic would take off {rec['recommended_off']['player_name']} ({rec['recommended_off']['reasons']}) and shortlist "
              f"{on['player_name']} (score {on['candidate_score']:.0f}; {on['evidence']}). Availability: {on['injury_status']}.")
limitation("A counterfactual shortlist under stated weights. It is not a forecast and cannot say a different "
           "change would have altered the result.")

In [ ]:
imp = ws.impact(); imp_t = imp[imp.team_name == FOCUS]
show("H-175", ws)
display(sa.impact_summary(imp_t))
interpret(f"Of {len(imp_t)} substitution decisions, {imp_t.contaminated.mean():.0%} have a goal, a red card or another own "
          f"change inside the before/after window and the opponent also changed in {imp_t.opponent_changed_in_window.mean():.0%}. "
          f"The median change in team xG per window is {imp_t.delta_team_xg.median():+.3f}.")
limitation("Before/after differences are confounded by score, opponent changes, fatigue and regression to the "
           "mean. They are not effects of the substitution.")

In [ ]:
with timed("substitution backtest"):
    sb = ws.sub_agreement()
rnd = (3 / sb.bench_size.clip(lower=3)).mean()
display(sb.head(8))
interpret(f"For {len(sb)} non-injury substitutions the player who actually came on was in the heuristic's top three "
          f"{sb.on_in_top_k.mean():.1%} of the time; picking three bench players at random would manage about {rnd:.0%}.")
show("H-178", ws)
show("H-179", ws, match_id=mid)